In [0]:
%pip install --upgrade pip setuptools wheel

# install compatible versions avoiding strict constraints
%pip install \
    langchain==0.2.11 \
    langchain-community==0.2.9 \
    langchain-openai==0.1.6 \
    chromadb==0.5.3 \
    openai==1.40.6 \
    tiktoken==0.7.0 \
    pydantic==2.6.4

dbutils.library.restartPython()



In [0]:
# rag_ingest_pipeline.py
from pyspark.sql import SparkSession
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import os, json

spark = SparkSession.builder.appName("RAG_Ingestion_Pipeline").getOrCreate()

# ----- CONFIG -----
JSON_PATH = "pratice_ai.ai_sch.json_volume"
VECTOR_DB_PATH = "pratice_ai.vector_store"
PROCESSED_LOG_PATH = "pratice_ai.processed_delta"
os.environ["OPENAI_API_KEY"] = "<YOUR_KEY>"

# ----- STEP 1: Identify new files -----
def get_new_files():
    all_files = [f.path for f in dbutils.fs.ls(JSON_PATH) if f.path.endswith(".json")]
    processed_files = []
    if spark._jsparkSession.catalog().tableExists(PROCESSED_LOG_PATH):
        processed_files = [r.file_name for r in spark.read.format("delta").load(PROCESSED_LOG_PATH).collect()]
    new_files = [f for f in all_files if f not in processed_files]
    return new_files, processed_files

# ----- STEP 2: Process JSONs -----
def load_jsons(files):
    if not files:
        return []
    df = spark.read.option("multiLine", True).json(files)
    df = df.selectExpr("id", "concat_ws(' ', title, description, content) as text")
    return [Document(page_content=row["text"], metadata={"id": row["id"]}) for row in df.collect()]

# ----- STEP 3: Embed & Store -----
def embed_and_store(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma(persist_directory=VECTOR_DB_PATH, embedding_function=embeddings)
    vectorstore.add_documents(chunks)
    vectorstore.persist()

# ----- STEP 4: Log processed files -----
def update_log(new_files, old_files):
    log_df = spark.createDataFrame([(f,) for f in new_files + old_files], ["file_name"])
    log_df.write.format("delta").mode("overwrite").save(PROCESSED_LOG_PATH)

# ----- MAIN FLOW -----
if __name__ == "__main__":
    new_files, old_files = get_new_files()
    if not new_files:
        print("✅ No new files to process.")
    else:
        print(f"🚀 Processing {len(new_files)} new JSON files...")
        docs = load_jsons(new_files)
        embed_and_store(docs)
        update_log(new_files, old_files)
        print("✅ Vector DB updated successfully.")
